In [ ]:
!nvidia-smi || true
!pip -q uninstall -y transformers
!pip -q install "transformers==4.57.1" "accelerate>=1.10.0" "bitsandbytes>=0.47.0" "sentencepiece" "huggingface_hub" "gradio"


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.
'true' is not recognized as an internal or external command,
operable program or batch file.


^C



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from huggingface_hub import login
from getpass import getpass

token = getpass("HF token: ")
login(token=token)
print("HF login complete")


HF token: ··········
HF login complete


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 36.5 MB/s eta 0:00:00


In [ ]:
!pip -q uninstall -y transformers accelerate bitsandbytes keras_nlp
!pip -q install --no-cache-dir "transformers" "accelerate" "bitsandbytes" "sentencepiece"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 237.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 427.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 224.2 MB/s eta 0:00:00


In [ ]:
# Cell 3: Load MedGemma (4-bit)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

MODEL_ID = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)

gen = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("Loaded:", MODEL_ID)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Device set to use cuda:0


Loaded: google/medgemma-4b-it


In [ ]:
import json, re, time

SYSTEM_PROMPT = """You are MedAssist, an AI clinical triage decision-support assistant for low-resource settings.

Safety rules:
1) This is decision support only and not a diagnosis.
2) Never provide medication dosages.
3) Never claim certainty or definitive diagnosis.
4) Prioritize life-threatening red flags.
5) Return ONLY one valid JSON object, no markdown, no extra prose.

Required JSON keys:
triage_summary, differential_diagnosis, urgency_level, red_flags,
recommended_next_steps, patient_friendly_explanation, limitations, disclaimer
"""

def build_prompt(payload):
    return f"""<system>
{SYSTEM_PROMPT}
</system>
<user>
Patient context:
{json.dumps(payload, indent=2)}

Return strict JSON only.
</user>
<assistant>"""

def extract_json(text):
    fence = re.search(r"```json\\s*([\\s\\S]*?)```", text, re.IGNORECASE)
    if fence:
        return json.loads(fence.group(1).strip())

    candidates = re.findall(r"\\{[\\s\\S]*\\}", text)
    for c in reversed(candidates):
        try:
            obj = json.loads(c)
            if isinstance(obj, dict):
                return obj
        except:
            pass
    raise ValueError("No JSON found")

def fallback_structured(payload, note):
    # conservative fallback for parser failures only
    return {
        "triage_summary": f"Model output parsing failed. {note}",
        "differential_diagnosis": [
            {"condition": "Acute cardiopulmonary concern", "rationale": "Chest pain + symptoms require urgent evaluation.", "confidence": "medium"},
            {"condition": "Acute coronary syndrome possibility", "rationale": "Radiating pain and sweating are concerning.", "confidence": "medium"},
            {"condition": "Other non-cardiac chest pain cause", "rationale": "Further clinical workup is required.", "confidence": "low"}
        ],
        "urgency_level": "EMERGENCY" if "chest pain" in payload.get("chief_complaint","").lower() else "HIGH",
        "red_flags": ["Severe symptom pattern"],
        "recommended_next_steps": [
            "Immediate clinician assessment",
            "Repeat vitals and focused exam",
            "Urgent referral/transfer if indicated"
        ],
        "patient_friendly_explanation": (
            "Your symptoms may be serious. Please seek urgent medical care now."
            if payload.get("patient_friendly_mode") else "Not requested."
        ),
        "limitations": "Generated with safety fallback due to non-JSON model output.",
        "disclaimer": "This tool is for decision support only and does not replace professional medical judgment."
    }

def run_case(payload):
    prompt = build_prompt(payload) + "\\n\\nSTRICT RULE: Output only one JSON object. No markdown."

    t0 = time.perf_counter()
    out = gen(
        prompt,
        do_sample=False,          # critical for stability
        max_new_tokens=220,
        return_full_text=False
    )[0]["generated_text"]

    parsed_by_model = True
    try:
        parsed = extract_json(out)
    except Exception:
        retry_prompt = prompt + "\\nReturn ONLY valid JSON object with required keys. Nothing else."
        out2 = gen(
            retry_prompt,
            do_sample=False,
            max_new_tokens=320,
            return_full_text=False
        )[0]["generated_text"]
        try:
            parsed = extract_json(out2)
        except Exception:
            parsed_by_model = False
            parsed = fallback_structured(payload, "No JSON produced by model after retries.")

    parsed["inference_time_seconds"] = round(time.perf_counter() - t0, 3)
    parsed["_parsed_by_model"] = parsed_by_model
    return parsed



In [ ]:
sample_case = {
  "age": 54,
  "gender": "Male",
  "chief_complaint": "Severe chest pain",
  "symptoms": "Crushing central chest pain radiating to left arm, sweating, mild shortness of breath",
  "duration": "40 minutes",
  "vitals": {
    "temperature": 36.9,
    "heart_rate": 118,
    "blood_pressure": "160/95",
    "respiratory_rate": 24,
    "oxygen_saturation": 93
  },
  "medical_history": "Hypertension, smoker",
  "medications": "Amlodipine",
  "patient_friendly_mode": True
}

result = run_case(sample_case)
result



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'triage_summary': 'Model output parsing failed. No JSON produced by model after retries.',
 'differential_diagnosis': [{'condition': 'Acute cardiopulmonary concern',
   'rationale': 'Chest pain + symptoms require urgent evaluation.',
   'confidence': 'medium'},
  {'condition': 'Acute coronary syndrome possibility',
   'rationale': 'Radiating pain and sweating are concerning.',
   'confidence': 'medium'},
  {'condition': 'Other non-cardiac chest pain cause',
   'rationale': 'Further clinical workup is required.',
   'confidence': 'low'}],
 'urgency_level': 'EMERGENCY',
 'red_flags': ['Severe symptom pattern'],
 'recommended_next_steps': ['Immediate clinician assessment',
  'Repeat vitals and focused exam',
  'Urgent referral/transfer if indicated'],
 'patient_friendly_explanation': 'Your symptoms may be serious. Please seek urgent medical care now.',
 'limitations': 'Generated with safety fallback due to non-JSON model output.',
 'disclaimer': 'This tool is for decision support only an

In [ ]:
import json
with open("sample_output.json", "w") as f:
    json.dump(result, f, indent=2)
print("saved sample_output.json")


saved sample_output.json


In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())


Saving test_cases.json to test_cases.json
dict_keys(['test_cases.json'])


In [ ]:
import pathlib
tc_path = pathlib.Path("test_cases.json")
print("exists:", tc_path.exists(), "| path:", tc_path.resolve())


exists: True | path: /content/test_cases.json


In [ ]:
import json, pathlib, time

# BOM-safe load
tc_path = pathlib.Path("test_cases.json")
cases = json.loads(tc_path.read_text(encoding="utf-8-sig"))

urgency_correct = 0
times = []
parsed_direct = 0
fallback_count = 0
redflag_hits = 0
redflag_total = 0

def has_expected_flag(pred_flags, expected_keywords):
    text = " ".join(pred_flags).lower()
    return any(k.lower() in text for k in expected_keywords)

print("Running 10-case evaluation...\n")

for c in cases:
    out = run_case(c["input"])

    pred = out.get("urgency_level")
    exp = c.get("expected_urgency")
    ok = (pred == exp)
    urgency_correct += int(ok)

    t = float(out.get("inference_time_seconds", 0.0))
    times.append(t)

    model_json = bool(out.get("_parsed_by_model"))
    parsed_direct += int(model_json)
    fallback_count += int(not model_json)

    expected_flags = c.get("expected_red_flags", [])
    if expected_flags:
        redflag_total += 1
        hit = has_expected_flag(out.get("red_flags", []), expected_flags)
        redflag_hits += int(hit)
    else:
        hit = True

    print(
        f"{c['id']} | {c['name']} | pred={pred} exp={exp} | "
        f"urgency_ok={ok} | redflag_ok={hit} | parsed_by_model={model_json} | t={t}s"
    )

total = len(cases)
avg_time = sum(times) / total if total else 0.0
redflag_rate = (redflag_hits / redflag_total) if redflag_total else 1.0

print("\n--- Summary ---")
print(f"Total cases: {total}")
print(f"Urgency correctness: {urgency_correct}/{total} ({urgency_correct/total:.2%})")
print(f"Parsed directly as JSON: {parsed_direct}/{total} ({parsed_direct/total:.2%})")
print(f"Fallback parse-rescue: {fallback_count}/{total} ({fallback_count/total:.2%})")
print(f"Red-flag detection hit-rate: {redflag_hits}/{redflag_total} ({redflag_rate:.2%})")
print(f"Average inference time: {avg_time:.3f}s")



Running 10-case evaluation...



NameError: name 'run_case' is not defined

In [ ]:
import gradio as gr
import json

def triage_demo(age, gender, chief, symptoms, duration, temp, hr, bp, rr, spo2, history, meds, pf):
    payload = {
        "age": int(age),
        "gender": gender,
        "chief_complaint": chief,
        "symptoms": symptoms,
        "duration": duration,
        "vitals": {
            "temperature": float(temp),
            "heart_rate": int(hr),
            "blood_pressure": bp,
            "respiratory_rate": int(rr),
            "oxygen_saturation": int(spo2),
        },
        "medical_history": history,
        "medications": meds,
        "patient_friendly_mode": bool(pf),
    }
    out = run_case(payload)
    return json.dumps(out, indent=2)

demo = gr.Interface(
    fn=triage_demo,
    inputs=[
        gr.Number(value=54, label="Age"),
        gr.Textbox(value="Male", label="Gender"),
        gr.Textbox(value="Severe chest pain", label="Chief Complaint"),
        gr.Textbox(value="Crushing central chest pain radiating to left arm, sweating, mild shortness of breath", label="Symptoms", lines=3),
        gr.Textbox(value="40 minutes", label="Duration"),
        gr.Number(value=36.9, label="Temperature"),
        gr.Number(value=118, label="Heart Rate"),
        gr.Textbox(value="160/95", label="Blood Pressure"),
        gr.Number(value=24, label="Respiratory Rate"),
        gr.Number(value=93, label="Oxygen Saturation"),
        gr.Textbox(value="Hypertension, smoker", label="Medical History"),
        gr.Textbox(value="Amlodipine", label="Current Medications"),
        gr.Checkbox(value=True, label="Patient-Friendly Mode"),
    ],
    outputs=gr.Code(language="json", label="Structured Triage Output"),
    title="MedAssist - MedGemma Colab Demo",
)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c102af68fd9ff577e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
